# Lesson 11 – Inference-Time Scaling for Creative AI

## Chapter 4 Connection

Chapter 4 of *Build a Reasoning Model from Scratch* focuses on **improving reasoning without retraining the model**.

That idea is called **inference-time scaling**.

Instead of changing the model's weights, we spend more compute while generating an answer.

In this lab, we will map the chapter's ideas onto **Weird AI**:

| Chapter 4 concept | Weird AI version |
|---|---|
| Ask the model to explain step by step | Ask the model for more structured lyrics |
| Temperature scaling | Control lyric randomness and creativity |
| Top-p sampling | Limit generation to likely-but-varied options |
| Self-consistency | Generate multiple candidates and pick the best |
| Accuracy improvement | Better parody output without retraining |

This notebook is a concept lab. The programming assignment is completed separately in `src/weird_ai/generator.py`.


## 1. What Is Inference-Time Scaling?

Inference-time scaling means improving output quality by spending more compute **after** the model has already been trained.

Examples:

- Generate a longer response.
- Ask for step-by-step reasoning.
- Generate several possible answers.
- Score or vote on the candidates.
- Let the model refine its own response.

The tradeoff is important:

> Better output may require more tokens, more latency, and more cost.


In [ ]:
# A tiny helper for this notebook

def count_tokens_roughly(text):
    """Very rough token estimate for classroom discussion."""
    return len(text.split())


short_answer = "The answer is 83."

long_answer = '''
Let's solve step by step.
Half the value of 3x - 9 is x + 37.
So (3x - 9) / 2 = x + 37.
Multiply both sides by 2.
3x - 9 = 2x + 74.
Subtract 2x from both sides.
x - 9 = 74.
Add 9 to both sides.
x = 83.
The answer is 83.
'''

print("Short answer rough token count:", count_tokens_roughly(short_answer))
print("Long answer rough token count:", count_tokens_roughly(long_answer))


## 2. Chain-of-Thought Prompting

Chapter 4 begins with a simple idea: ask the model to reason step by step.

For math and logic tasks, this can improve accuracy because the model produces intermediate reasoning before the final answer.

For Weird AI, the same pattern can be adapted:

Instead of asking for hidden reasoning, we can ask for **explicit creative structure**.

Examples:

- "Write 8 lines."
- "Use an AABB rhyme pattern."
- "Keep the tone dramatic and funny."
- "End each pair of lines with rhyming words."

This gives the model more structure during generation.


In [ ]:
basic_prompt = "Write a parody song about database indexes."

structured_prompt = '''
Write an 8-line emo parody song about database indexes.

Requirements:
- Use short lyric lines.
- Use an AABB rhyme pattern if possible.
- Make it funny but still sound dramatic.
- Return only the lyrics.
'''

print("Basic prompt:")
print(basic_prompt)

print("\nStructured prompt:")
print(structured_prompt)


### Exercise

Write a structured prompt for a Weird AI parody about one of these topics:

- recursion
- Docker containers
- finals week
- merge conflicts
- SQL joins

Your prompt should include at least three constraints.


In [ ]:
# Write your structured prompt here

my_prompt = '''
TODO: Replace this with your structured prompt.
'''

print(my_prompt)


## 3. Temperature and Creativity

Temperature controls how strongly the model favors the most likely next token.

Conceptually:

- Low temperature: safer, more predictable, less creative
- Medium temperature: balanced
- High temperature: more surprising, more creative, more likely to become incoherent

We can explore the idea using a small fake vocabulary.


In [ ]:
import math
import random

tokens = ["rain", "pain", "database", "index", "heart", "query"]
logits = [5.0, 4.7, 2.0, 1.8, 4.2, 1.0]

def softmax(values):
    exps = [math.exp(v) for v in values]
    total = sum(exps)
    return [v / total for v in exps]

def apply_temperature(logits, temperature):
    return [value / temperature for value in logits]

for temp in [0.3, 1.0, 2.0]:
    scaled = apply_temperature(logits, temp)
    probs = softmax(scaled)
    print(f"\nTemperature: {temp}")
    for token, prob in zip(tokens, probs):
        print(f"{token:10s} {prob:.3f}")


## 4. Sampling from a Distribution

Greedy decoding always chooses the highest-probability token.

Sampling allows the model to choose from several likely tokens, which creates variety.

This is especially useful for creative generation.


In [ ]:
def sample_token(tokens, probabilities):
    return random.choices(tokens, weights=probabilities, k=1)[0]


for temp in [0.3, 1.0, 2.0]:
    scaled = apply_temperature(logits, temp)
    probs = softmax(scaled)
    samples = [sample_token(tokens, probs) for _ in range(12)]
    print(f"Temperature {temp}:")
    print(samples)
    print()


### Discussion

Why might high temperature be useful for parody lyrics but risky for math answers?


## 5. Top-p Sampling

Top-p sampling, also called nucleus sampling, keeps only the smallest set of tokens whose cumulative probability reaches a threshold.

For example:

- `top_p = 0.9` keeps enough likely tokens to cover 90% of the probability mass.
- Very unlikely tokens are removed before sampling.

This can preserve creativity while avoiding very low-quality choices.


In [ ]:
def top_p_filter(tokens, probabilities, top_p):
    pairs = sorted(zip(tokens, probabilities), key=lambda pair: pair[1], reverse=True)

    kept = []
    cumulative = 0.0

    for token, probability in pairs:
        kept.append((token, probability))
        cumulative += probability
        if cumulative >= top_p:
            break

    kept_tokens = [token for token, _ in kept]
    kept_probs = [prob for _, prob in kept]

    # Renormalize probabilities after filtering
    total = sum(kept_probs)
    kept_probs = [prob / total for prob in kept_probs]

    return kept_tokens, kept_probs


probs = softmax(apply_temperature(logits, 1.0))

for p in [0.5, 0.8, 0.95]:
    kept_tokens, kept_probs = top_p_filter(tokens, probs, p)
    print(f"top_p={p}")
    print(list(zip(kept_tokens, [round(prob, 3) for prob in kept_probs])))
    print()


## 6. From Self-Consistency to Best-of-N

In the chapter, self-consistency means:

1. Generate several answers.
2. Extract the final answer from each.
3. Pick the most common answer.

That works well for math because there is a correct final answer.

For Weird AI, there may not be a single repeated answer. Instead, we can adapt the idea:

1. Generate several parody candidates.
2. Evaluate each candidate.
3. Pick the highest-scoring candidate.

This is **best-of-N generation**.


In [ ]:
candidates = [
    {
        "name": "Candidate A",
        "text": "My query broke apart\nLike indexes in the dark",
        "rhyme_score": 0.70,
        "structure_score": 0.80,
        "syllable_score": 0.60,
    },
    {
        "name": "Candidate B",
        "text": "I searched my heart at night\nThe index made it right",
        "rhyme_score": 0.95,
        "structure_score": 0.90,
        "syllable_score": 0.85,
    },
    {
        "name": "Candidate C",
        "text": "Database sadness runs extremely complicated tonight",
        "rhyme_score": 0.10,
        "structure_score": 0.30,
        "syllable_score": 0.20,
    },
]

for candidate in candidates:
    candidate["overall_score"] = (
        candidate["rhyme_score"]
        + candidate["structure_score"]
        + candidate["syllable_score"]
    ) / 3

best = max(candidates, key=lambda candidate: candidate["overall_score"])

for candidate in candidates:
    print(candidate["name"], "score:", round(candidate["overall_score"], 2))

print("\nBest candidate:", best["name"])
print(best["text"])


## 7. The New Weird AI Pipeline

The assignment adds a new module:

```text
src/weird_ai/generator.py
```

It should not replace the older `generation.py` file. That file contains lower-level text generation utilities from the earlier lesson.

The new `generator.py` file represents a higher-level inference pipeline:

```text
Prompt
  ↓
Generate N Candidates
  ↓
Evaluate Each Candidate
  ↓
Rank Candidates
  ↓
Return Best Candidate
```

This turns Weird AI from a simple text generator into a more realistic AI application.


## 8. Mini Design Activity

Before you write code, design your best-of-N strategy.

Answer these questions:

1. How many candidates should Weird AI generate by default?
2. What temperature seems reasonable for creative lyrics?
3. Should `top_p` be used? Why or why not?
4. Which evaluation score should be the main ranking score?
5. What should happen if all candidates score poorly?


In [ ]:
# Write your design notes here as comments.

# Default number of candidates:
# Temperature:
# top_p:
# Main ranking score:
# Low-quality fallback strategy:


## 9. Reflection

1. Why can inference-time scaling improve output without retraining the model?
2. What is the cost of generating multiple candidates?
3. How does temperature affect creativity?
4. Why does Weird AI use best-of-N scoring instead of majority voting?
5. How does this lesson build on the evaluation pipeline from the previous lesson?
